# Exercise 2 — O2DPG workflow with the hybrid generator

The **hybrid generator** combines several generators configured in **one JSON file**.
The file used here (`config/hybrid_pp_cocktail.json`) defines two entries that are used alternately:

1. plain `pythia8pp` (**no cocktail**);
2. a **cocktail** of `pythia8pp` + `boxgen` injecting 5 Λc⁺ into the same event.

With `"fractions": [1, 1]`: one plain event, then one cocktail event, and so on.

Then you run the **full MC chain up to the AOD** with the two O2DPG steps:
1. `o2dpg_sim_workflow.py` → creates `workflow.json` (a graph of tasks);
2. `o2_dpg_workflow_runner.py` → runs it.

> Requires a valid AliEn token (see `00_setup.ipynb`).

In [ ]:
from swan_env import *

In [ ]:
ex2 = workdir("ex2")
os.environ["EX2"] = str(ex2)
import shutil
shutil.copy(PROJECT / "config" / "hybrid_pp_cocktail.json", ex2)
print((ex2 / "hybrid_pp_cocktail.json").read_text())

Notes on the JSON:
- `pythia8pp` needs no `config` block. `boxgen`, `pythia8`, `external` and `hepmc` must have one.
- A `cocktail` is **one** entry in `fractions`: `[1, 0]` gives only plain events, `[0, 1]` only cocktail events.
- A template can be generated with `$O2DPG_ROOT/MC/bin/o2_hybrid_gen.py --gen pythia8pp boxgen --output template.json`.

## 2a. Quick check with `o2-sim` only

Always test a new configuration with a few events and `--noGeant` first:

In [ ]:
%%bash
cd $EX2
o2exec o2-sim -n 4 -g hybrid --noGeant -o hybtest \
   --configKeyValues "GeneratorHybrid.configFile=$EX2/hybrid_pp_cocktail.json" > hybtest.log 2>&1
grep -iE 'hybrid' hybtest.log | head -20
o2exec root -l -b -q "$SIMTUT_PROJECT/macros/kineDaughters.C(\"hybtest_Kine.root\",4122)" | grep '^event'

**❓** Do the Λc⁺ appear only in every second event, as expected? (Λc⁺ does not decay here because there is no transport.)

## 2b. Create the workflow

- `-gen hybrid` + `-confKey "GeneratorHybrid.configFile=..."`: the generator setup.
- `-eCM 14000`: matches the energy of `pythia8pp`.
- `-tf 1 -ns 20`: 1 timeframe with 20 signal events.
- `-interactionRate 500000`: 500 kHz pp.
- `-run 302000`: a pre-defined pp MC run number (for CCDB conditions).
- `-j 4`: 4 transport workers (default 8 = GRID node).

In [ ]:
%%bash
cd $EX2
o2exec bash -c '${O2DPG_ROOT}/MC/bin/o2dpg_sim_workflow.py \
    -eCM 14000 -col pp -gen hybrid \
    -confKey "GeneratorHybrid.configFile=${SIMTUT_DIR}/ex2/hybrid_pp_cocktail.json" \
    -tf 1 -ns 20 -j 4 \
    -interactionRate 500000 -run 302000 -seed 42' 2>&1 | tail -5
ls -lh workflow.json

> 💡 For official productions, put the settings in an **ini file** instead of `-confKey`:
> ```ini
> [GeneratorHybrid]
> configFile = /path/to/hybrid_pp_cocktail.json
> ```
> and use `-gen hybrid -ini /path/to/file.ini`.

Look at the tasks in the workflow:

In [ ]:
%%bash
cd $EX2
o2exec bash -c '${O2DPG_ROOT}/MC/bin/o2_dpg_workflow_runner.py -f workflow.json --list-tasks' | head -60

## 2c. Run the workflow

**Step 1**: generation (`sgngen_1`) and transport (`sgnsim_1`) only. This takes a few minutes.

In [ ]:
%%bash
cd $EX2
time o2exec bash -c '${O2DPG_ROOT}/MC/bin/o2_dpg_workflow_runner.py -f workflow.json -tt sgnsim --cpu-limit 4 --mem-limit 14000' > runner_sim.log 2>&1
tail -5 runner_sim.log

**Step 2**: continue up to the AOD. Tasks that already finished are **not** run again (checkpointing).
This takes 15–30 minutes, so it runs **in the background**. Run the `status` cell to follow it. You can keep working meanwhile.

In [ ]:
start("${O2DPG_ROOT}/MC/bin/o2_dpg_workflow_runner.py -f workflow.json -tt aod --cpu-limit 4 --mem-limit 14000",
      cwd=ex2, log="runner_aod.log")

In [ ]:
status(ex2, "runner_aod.log")

While it runs:
- every task writes its own log, e.g. `tf1/sgnsim_1.log`, `tf1/tpcdigi_1.log`, …
- `pipeline_metric_*.log` holds CPU and memory measurements of each task.

> 💡 Optional: the **new modular runner** is selected with `ALIEN_O2DPG_WORKFLOW_RUNNER=new` and offers `--scheduler-policy timeframe|critical-path|best-fit`.

In [ ]:
%%bash
cd $EX2
ls tf1/*.log | head -40

## 2d. Look at the AOD

When the runner prints `Pipeline done success`:

In [ ]:
%%bash
cd $EX2
AOD=$(find . -name 'AO2D*.root' | head -1); echo "AOD: $AOD"
o2exec root -l -b -q "$SIMTUT_PROJECT/macros/aodMcSummary.C(\"$AOD\",4122)"

**❓ Questions**
1. How many MC collisions are in the AOD? Is that more than the 20 signal events? (Hint: interaction rate and timeframe length.)
2. How many Λc⁺ per MC collision? Is this consistent with 5 Λc⁺ in every second event plus the Pythia8 production?
3. Change `"fractions"` to `[3, 1]` and rerun from the generation step (`--rerun-from sgngen`). How does the Λc⁺ yield change?
4. **Bonus**: add `"mode": "parallel"` at the top level of the JSON and `GeneratorHybrid.num_workers=4` in `-confKey`. Compare the generation time (`tf1/sgngen_1.log`).

In [ ]:
%%bash
mkdir -p ~/simtutorial_results/ex2
cd $EX2 && cp $(find . -name 'AO2D*.root') workflow.json hybrid_pp_cocktail.json ~/simtutorial_results/ex2/ && ls ~/simtutorial_results/ex2

Next: **`03_anchored_external.ipynb`**